In [2]:
# repositories
!git clone https://github.com/Luffy65/Semantic-Correspondence.git # Clone repo
!git clone https://github.com/facebookresearch/dinov3.git # DINOv3
!pip install git+https://github.com/facebookresearch/segment-anything.git # SAM

# Install requirements (requirements.txt)
!pip install -r Semantic-Correspondence/requirements.txt
!pip install -r dinov3/requirements.txt

Cloning into 'Semantic-Correspondence'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 109 (delta 40), reused 87 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 4.68 MiB | 17.13 MiB/s, done.
Resolving deltas: 100% (40/40), done.
Cloning into 'dinov3'...
remote: Enumerating objects: 538, done.
remote: Counting objects: 100% (363/363), done.
remote: Compressing objects: 100% (264/264), done.
remote: Total 538 (delta 201), reused 99 (delta 99), pack-reused 175 (from 1)
Receiving objects: 100% (538/538), 9.88 MiB | 24.86 MiB/s, done.
Resolving deltas: 100% (223/223), done.
  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-are65wsv
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-are65wsv
  Resolved https://github.com/facebookresearch/segment-any

In [ ]:
# Dependencies
import os
import shutil
import gzip
import cv2
import json
from PIL import Image
import numpy as np
from google.colab import drive
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [ ]:
# Connect google drive, load and unzip data
# 1. Mount Drive
drive.mount('/content/drive')

# 2. Define Paths
DRIVE_ROOT = '/content/drive/MyDrive/AML-PROJECT-DATA/'
DATASET_ROOT = os.path.join(DRIVE_ROOT, 'dataset/')
DATASET_ARCHIVE = os.path.join(DATASET_ROOT, 'SPair-71k.tar.gz')
LOCAL_DATA_DIR = '/content/data'

# 3. Copy and Extract
if not os.path.exists(LOCAL_DATA_DIR):
    print(f"Extracting {DATASET_ARCHIVE} to local VM...")
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

    # shutil works for .zip, .tar, .tar.gz, etc.
    # format='gztar' explicitly tells it to handle gzip compression
    shutil.unpack_archive(DATASET_ARCHIVE, LOCAL_DATA_DIR, format='gztar')

    print("Done! Data is ready at:", LOCAL_DATA_DIR)
else:
    print("Data already loaded.")

Mounted at /content/drive
Extracting /content/drive/MyDrive/AML-PROJECT-DATA/dataset/SPair-71k.tar.gz to local VM...
Done! Data is ready at: /content/data


In [ ]:
# Setup and verify paths

SPAIR_ROOT = os.path.join(LOCAL_DATA_DIR, 'SPair-71k')

pair_ann_path = os.path.join(SPAIR_ROOT, 'PairAnnotation')
layout_path = os.path.join(SPAIR_ROOT, 'Layout')
image_path = os.path.join(SPAIR_ROOT, 'JPEGImages')

# Verify that paths exist
print(f"Verifying paths...")
print(f"  SPAIR_ROOT: {SPAIR_ROOT} → {'✓' if os.path.exists(SPAIR_ROOT) else '✗ DOES NOT EXIST'}")
print(f"  Layout: {layout_path} → {'✓' if os.path.exists(layout_path) else '✗ DOES NOT EXIST'}")
print(f"  PairAnnotation: {pair_ann_path} → {'✓' if os.path.exists(pair_ann_path) else '✗ DOES NOT EXIST'}")
print(f"  JPEGImages: {image_path} → {'✓' if os.path.exists(image_path) else '✗ DOES NOT EXIST'}")

# Verify that the trn.txt file exists
trn_file = os.path.join(layout_path, 'large', 'trn.txt')
print(f"  trn.txt: {trn_file} → {'✓' if os.path.exists(trn_file) else '✗ DOES NOT EXIST'}")

In [5]:
# Instantiate models
from segment_anything import SamPredictor, sam_model_registry

DINOV3_REPO_DIR = "dinov3"
CHECKPOINTS_ROOT = os.path.join(DRIVE_ROOT, 'checkpoints/')
SAM_WEIGHTS_PATH = os.path.join(CHECKPOINTS_ROOT, 'sam_vit_h_4b8939.pth')
DINOV3_WEIGHTS_PATH = os.path.join(CHECKPOINTS_ROOT, 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth')

sam = sam_model_registry["default"](checkpoint=SAM_WEIGHTS_PATH)

sampredictor = SamPredictor(sam)

dinov2_vitb14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
dinov3_vitb16 = torch.hub.load(DINOV3_REPO_DIR, 'dinov3_vitb16', source='local', weights=DINOV3_WEIGHTS_PATH) # DINOv3 ViT model pretrained on web images

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:01<00:00, 334MB/s]


Downloading: "file:///content/drive/.shortcut-targets-by-id/1fEWpONVft365O47IhEDLKZ2a0WkAuhyP/AML-PROJECT-DATA/checkpoints/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth" to /root/.cache/torch/hub/checkpoints/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth


100%|██████████| 327M/327M [00:10<00:00, 32.7MB/s]


In [ ]:
# Print models
print('======================models======================')
print(sam)
print('----------------------\n')
print(dinov2_vitb14)
print('----------------------\n')
print(dinov3_vitb16)
print('======================END print models======================')

======================models======================
Sam(
  (image_encoder): ImageEncoderViT(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 1280, kernel_size=(16, 16), stride=(16, 16))
    )
    (blocks): ModuleList(
      (0-31): 32 x Block(
        (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=1280, out_features=3840, bias=True)
          (proj): Linear(in_features=1280, out_features=1280, bias=True)
        )
        (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (lin1): Linear(in_features=1280, out_features=5120, bias=True)
          (lin2): Linear(in_features=5120, out_features=1280, bias=True)
          (act): GELU(approximate='none')
        )
      )
    )
    (neck): Sequential(
      (0): Conv2d(1280, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): LayerNorm2d()
      (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1),

In [ ]:
def setup_model_for_finetuning(model, model_type, head, num_layers_to_unfreeze=2):
    """
    Freeze all layers except the last few

    Args:
        model: the model to finetune (dinov2_vitb14 , dinov3_vitb16 or SAM)
        model_type: 'dinov2', 'dinov3' or 'sam'
        num_layers_to_unfreeze: Number of last blocks to unfreeze
        head: if True, unfreeze norm layer for dino, neck for SAM
    """
    # First, freeze everything
    for param in model.parameters():
        param.requires_grad = False

    if 'dinov2' in model_type or 'dinov3' in model_type:
        # DINO: Unfreeze last transformer blocks
        total_blocks = len(model.blocks)
        print(f"Total DINO blocks: {total_blocks}")
        print(f"Unfreezing last {num_layers_to_unfreeze} blocks...")

        for i in range(total_blocks - num_layers_to_unfreeze, total_blocks):
            for param in model.blocks[i].parameters():
                param.requires_grad = True

        if head:
            # Also unfreeze the final norm layer
            for param in model.norm.parameters():
                param.requires_grad = True

    elif 'sam' in model_type:
        # SAM: Unfreeze last layers of image encoder
        encoder = model.image_encoder

        if head:
            # Unfreeze neck (final convolutions)
            for param in encoder.neck.parameters():
                param.requires_grad = True

        # Optionally unfreeze last few blocks

        total_blocks = len(model.image_encoder.blocks)
        print(f"Total SAM blocks: {total_blocks}")
        print(f"Unfreezing last {num_layers_to_unfreeze} blocks + neck...")

        for i in range(total_blocks - num_layers_to_unfreeze, total_blocks):
            for param in model.image_encoder.blocks[i].parameters():
                param.requires_grad = True

    # Count trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"Trainable parameters: {trainable_params:,} / {total_params:,} "
          f"({100 * trainable_params / total_params:.2f}%)")

    return model

Choose the model and configuration:
- **model**: 'dinov2', 'dinov3', 'sam'
- **config**: 1 (norm/neck only), 2 (norm/neck + last 2 blocks)

In [ ]:
# # --- 3. Training Logic ---

def extract_descriptors(model, img, model_type, patch_size):
    """
    Extract dense features from the model
    Args:
        model: DINO or SAM model
        img: Input image tensor (B, 3, H, W) - already normalized by dataset
        model_type: 'dinov2' or 'sam'
        patch_size: Patch size of the model
    Returns:
        features: (B, H_feat, W_feat, C)
    """
    if 'dinov2' in model_type or 'dinov3' in model_type:
        res = model.forward_features(img)
        x = res['x_norm_patchtokens']
        B, N, C = x.shape
        H = img.shape[2] // patch_size
        W = img.shape[3] // patch_size
        x = x.reshape(B, H, W, C)
        return x

    elif 'sam' in model_type:
        # SAM expects images in range [0, 1], but your dataset normalizes to ImageNet stats
        # You may need to denormalize first if SAM doesn't work with normalized images
        x = model.image_encoder(img)
        x = x.permute(0, 2, 3, 1)  # (B, C, H, W) -> (B, H, W, C)
        return x

    return None

def train_epoch(model, train_loader, optimizer, device, model_type, patch_size, debug_batches=None):
    """
    Train for one epoch using cross-entropy loss on similarity maps

    Args:
        debug_batches: if specified, process only the first N batches (for fast debug)
    """
    model.train()
    total_loss = 0
    num_batches = 0

    for batch_idx, batch in enumerate(train_loader):
        # === DEBUG: STOP AFTER N BATCHES ===
        if debug_batches is not None and batch_idx >= debug_batches:
            print(f"    [DEBUG] Train stopped at {debug_batches} batches")
            break

        src_img = batch['src_img'].to(device)  # (B, 3, H, W) Already normalized
        trg_img = batch['trg_img'].to(device)
        src_kps_list = batch['src_kps']  # List[Tensor(N_i, 2)]
        trg_kps_list = batch['trg_kps']  # List[Tensor(N_i, 2)]

        optimizer.zero_grad()

        # Extract dense features
        src_feat = extract_descriptors(model, src_img, model_type, patch_size)
        trg_feat = extract_descriptors(model, trg_img, model_type, patch_size)

        B, Hf, Wf, C = src_feat.shape
        H_img, W_img = src_img.shape[2], src_img.shape[3]  # Current image size (224x224)

        # Normalize features for cosine similarity
        src_feat = F.normalize(src_feat, dim=-1)
        trg_feat = F.normalize(trg_feat, dim=-1)

        # Compute loss for all keypoints in the batch
        batch_loss = 0
        num_valid_kps = 0

        for b in range(B):
            # Get keypoints for this image
            src_kps = src_kps_list[b].to(device)  # (N_i, 2)
            trg_kps = trg_kps_list[b].to(device)  # (N_i, 2)

            num_kps = src_kps.shape[0]  # Number of keypoints for this pair

            for kp_idx in range(num_kps):
                src_kp = src_kps[kp_idx]
                trg_kp = trg_kps[kp_idx]

                # Skip invalid keypoints (check for NaN or negative values)
                if torch.isnan(src_kp).any() or torch.isnan(trg_kp).any():
                    continue
                if src_kp[0] < 0 or src_kp[1] < 0 or trg_kp[0] < 0 or trg_kp[1] < 0:
                    continue

                # Map source keypoint to feature grid
                feat_x = int((src_kp[0] / W_img) * Wf)
                feat_y = int((src_kp[1] / H_img) * Hf)
                feat_x = max(0, min(feat_x, Wf - 1))
                feat_y = max(0, min(feat_y, Hf - 1))

                # Extract source descriptor
                src_desc = src_feat[b, feat_y, feat_x, :]  # (C,)

                # Compute similarity with ALL target locations
                trg_feat_flat = trg_feat[b].reshape(Hf * Wf, C)  # (H*W, C)
                sim = torch.matmul(trg_feat_flat, src_desc)  # (H*W,)

                # Ground truth: target keypoint location in feature grid
                trg_feat_x = int((trg_kp[0] / W_img) * Wf)
                trg_feat_y = int((trg_kp[1] / H_img) * Hf)
                trg_feat_x = max(0, min(trg_feat_x, Wf - 1))
                trg_feat_y = max(0, min(trg_feat_y, Hf - 1))

                gt_idx = trg_feat_y * Wf + trg_feat_x

                # Cross-entropy loss
                kp_loss = F.cross_entropy(
                    sim.unsqueeze(0),
                    torch.tensor([gt_idx], device=device)
                )

                batch_loss += kp_loss
                num_valid_kps += 1

        # Average loss over valid keypoints
        if num_valid_kps > 0:
            batch_loss = batch_loss / num_valid_kps
            batch_loss.backward()
            optimizer.step()

            total_loss += batch_loss.item()
            num_batches += 1

        # More frequent print if in debug mode
        print_freq = 5 if debug_batches else 100
        if batch_idx % print_freq == 0:
            loss_val = batch_loss.item() if num_valid_kps > 0 else 0.0
            print(f"    Batch {batch_idx}/{len(train_loader) if not debug_batches else debug_batches} | Loss: {loss_val:.4f} | Valid KPs: {num_valid_kps}")

    avg_loss = total_loss / num_batches if num_batches > 0 else 0
    return avg_loss



def validate_epoch(model, val_loader, device, model_type, patch_size,
                   thresholds=[0.1], debug_batches=None):
    """
    debug_batches: if specified, process only the first N batches
    """
    model.eval()

    correct_counts = {t: 0 for t in thresholds}
    total_kps = 0

    with torch.no_grad():
        for batch_idx, batch in enumerate(val_loader):
            # STOP after N batches for debug
            if debug_batches is not None and batch_idx >= debug_batches:
                print(f"    [DEBUG] Validation stopped at {debug_batches} batches")
                break

            src_img = batch['src_img'].to(device)
            trg_img = batch['trg_img'].to(device)
            src_kps_list = batch['src_kps']
            trg_kps_list = batch['trg_kps']
            trg_bbox = batch['trg_bbox']

            B = src_img.shape[0]

            # Extract features
            src_feat = extract_descriptors(model, src_img, model_type, patch_size)
            trg_feat = extract_descriptors(model, trg_img, model_type, patch_size)

            # Normalize
            src_feat = F.normalize(src_feat, dim=-1)
            trg_feat = F.normalize(trg_feat, dim=-1)

            Hf, Wf, C = src_feat.shape[1], src_feat.shape[2], src_feat.shape[3]
            H_img, W_img = src_img.shape[2], src_img.shape[3]

            # Process each image in batch
            for b in range(B):
                src_kps = src_kps_list[b].to(device)
                trg_kps = trg_kps_list[b].to(device)

                num_kps = src_kps.shape[0]

                # Compute norm factor
                bbox = trg_bbox[b]
                bbox_w = bbox[2] - bbox[0]
                bbox_h = bbox[3] - bbox[1]
                norm_factor = max(bbox_w, bbox_h).item()

                # Predict keypoints
                for kp_idx in range(num_kps):
                    src_kp = src_kps[kp_idx]
                    trg_kp = trg_kps[kp_idx]

                    # Skip invalid
                    if torch.isnan(src_kp).any() or torch.isnan(trg_kp).any():
                        continue
                    if src_kp[0] < 0 or src_kp[1] < 0 or trg_kp[0] < 0 or trg_kp[1] < 0:
                        continue

                    # Map to feature grid
                    feat_x = int((src_kp[0] / W_img) * Wf)
                    feat_y = int((src_kp[1] / H_img) * Hf)
                    feat_x = max(0, min(feat_x, Wf - 1))
                    feat_y = max(0, min(feat_y, Hf - 1))

                    # Extract descriptor
                    src_desc = src_feat[b, feat_y, feat_x, :]

                    # Find best match
                    trg_feat_flat = trg_feat[b].reshape(Hf * Wf, C)
                    sim = torch.matmul(trg_feat_flat, src_desc)

                    best_idx = torch.argmax(sim)
                    pred_y = (best_idx // Wf).item()
                    pred_x = (best_idx % Wf).item()

                    # Map back to image coords
                    pred_kp_x = pred_x * (W_img / Wf)
                    pred_kp_y = pred_y * (H_img / Hf)

                    # Compute distance
                    dist = torch.sqrt(
                        (pred_kp_x - trg_kp[0])**2 +
                        (pred_kp_y - trg_kp[1])**2
                    ).item()

                    # Check thresholds
                    for alpha in thresholds:
                        threshold = alpha * norm_factor
                        if dist <= threshold:
                            correct_counts[alpha] += 1

                    total_kps += 1

            if batch_idx % 2 == 0:  # More frequent print
                print(f"    Val Batch {batch_idx}")

    # Compute PCK
    pck_results = {}
    for alpha in thresholds:
        pck = 100.0 * correct_counts[alpha] / total_kps if total_kps > 0 else 0
        pck_results[alpha] = pck

    return pck_results

In [ ]:
def read_img(path):
    img = np.array(Image.open(path).convert('RGB'))
    return torch.tensor(img.transpose(2, 0, 1).astype(np.float32))

class SPairDataset(Dataset):
    def __init__(self, pair_ann_path, layout_path, image_path, dataset_size, pck_alpha, datatype):
        self.datatype = datatype
        self.pck_alpha = pck_alpha
        ann_list = os.path.join(layout_path, dataset_size, datatype + '.txt')
        self.ann_files = [x.strip() for x in open(ann_list, "r").readlines() if x.strip()]
        self.pair_ann_path = pair_ann_path
        self.image_path = image_path

    def __len__(self):
        return len(self.ann_files)

    def __getitem__(self, idx):
      ann_file = self.ann_files[idx] + '.json'
      with open(os.path.join(self.pair_ann_path, self.datatype, ann_file)) as f:
          annotation = json.load(f)

      category = annotation['category']
      src_img = read_img(os.path.join(self.image_path, category, annotation['src_imname']))
      trg_img = read_img(os.path.join(self.image_path, category, annotation['trg_imname']))

      # ========== RESIZE ==========
      TARGET_H, TARGET_W = 224, 224

      # Save original dimensions
      orig_h_src, orig_w_src = src_img.shape[1], src_img.shape[2]
      orig_h_trg, orig_w_trg = trg_img.shape[1], trg_img.shape[2]

      # Calculate scale factors
      scale_x_src = TARGET_W / orig_w_src
      scale_y_src = TARGET_H / orig_h_src
      scale_x_trg = TARGET_W / orig_w_trg
      scale_y_trg = TARGET_H / orig_h_trg

      # Resize images
      src_img = F.interpolate(
          src_img.unsqueeze(0),
          size=(TARGET_H, TARGET_W),
          mode='bilinear',
          align_corners=False
      ).squeeze(0).clone()

      trg_img = F.interpolate(
          trg_img.unsqueeze(0),
          size=(TARGET_H, TARGET_W),
          mode='bilinear',
          align_corners=False
      ).squeeze(0).clone()

      # Normalize (0-255 -> 0-1 -> ImageNet normalization)
      mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
      std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
      src_img = (src_img / 255.0 - mean) / std
      trg_img = (trg_img / 255.0 - mean) / std

      # Scale keypoints
      src_kps = torch.tensor(annotation['src_kps'], dtype=torch.float32)
      src_kps[:, 0] *= scale_x_src
      src_kps[:, 1] *= scale_y_src

      trg_kps = torch.tensor(annotation['trg_kps'], dtype=torch.float32)
      trg_kps[:, 0] *= scale_x_trg
      trg_kps[:, 1] *= scale_y_trg

      # ========== BBOX AND PCK THRESHOLD ==========
      trg_bbox = annotation['trg_bndbox']  # [xmin, ymin, xmax, ymax]

      # Scale the bbox to the new dimensions (consistent with resized image)
      trg_bbox_scaled = [
          trg_bbox[0] * scale_x_trg,
          trg_bbox[1] * scale_y_trg,
          trg_bbox[2] * scale_x_trg,
          trg_bbox[3] * scale_y_trg
      ]

      # Calculate PCK threshold on the scaled bbox
      pck_threshold = max(
          trg_bbox_scaled[2] - trg_bbox_scaled[0],
          trg_bbox_scaled[3] - trg_bbox_scaled[1]
      ) * self.pck_alpha

      return {
          'src_img': src_img,              # (3, 224, 224)
          'trg_img': trg_img,              # (3, 224, 224)
          'src_kps': src_kps,              # (N, 2) scaled
          'trg_kps': trg_kps,              # (N, 2) scaled
          'trg_bbox': torch.tensor(trg_bbox_scaled, dtype=torch.float32),  # (4,)
          'pck_threshold': pck_threshold,   # Scalar float
          'category': annotation.get('category', ''),
          'src_imsize': (orig_h_src, orig_w_src),  # Original dimensions
          'trg_imsize': (orig_h_trg, orig_w_trg)
      }

# Quick test
print("SPairDataset class redefined. Testing...")

Classe SPairDataset ridefinita. Testando...


In [ ]:
def custom_collate_fn(batch):
    """
    Manage batch with variable number of keypoints.
    """
    src_imgs = torch.stack([item['src_img'] for item in batch])
    trg_imgs = torch.stack([item['trg_img'] for item in batch])

    # Keypoints: list (variable number)
    src_kps = [item['src_kps'] for item in batch]
    trg_kps = [item['trg_kps'] for item in batch]

    # Bbox and threshold: can be stacked (fixed dimension)
    trg_bbox = torch.stack([item['trg_bbox'] for item in batch])  # (B, 4)
    pck_threshold = torch.tensor([item['pck_threshold'] for item in batch])  # (B,)

    categories = [item['category'] for item in batch]

    return {
        'src_img': src_imgs,
        'trg_img': trg_imgs,
        'src_kps': src_kps,
        'trg_kps': trg_kps,
        'trg_bbox': trg_bbox,
        'pck_threshold': pck_threshold,
        'category': categories
    }

In [ ]:
# Performs the fine-tuning of the model.
def finetune(
    model,
    pair_ann_path,
    layout_path,
    image_path,
    model_type,
    patch_size,
    head,
    learning_rate,
    num_layers_to_unfreeze,
    dataset_size='large',
    pck_alpha=0.1,
    num_epochs=5,
    batch_size=8,
    device='cuda',
    debug_batches=None # None=full, N=only N batches
):
    """
    debug_batches:
      - None: full epoch
      - N: process only N batches for train and N/2 for val (very fast!)
    """
    print(f"\n{'='*70}")
    print(f"Starting Light Fine-tuning")
    print(f"{'='*70}")
    print(f"Model: {model_type}")
    print(f"Layers to unfreeze: {num_layers_to_unfreeze}")
    print(f"Head (Norm/Neck): {head}")
    print(f"Epochs: {num_epochs}")
    print(f"Batch size: {batch_size}")
    print(f"Learning rate: {learning_rate}")

    # Show if in debug mode
    if debug_batches:
        print(f"⚠️  DEBUG MODE: {debug_batches} train batch, {debug_batches//2} val batch")

    print(f"{'='*70}\n")

    # Setup model
    model = setup_model_for_finetuning(model, model_type, num_layers_to_unfreeze, head)
    model = model.to(device)

    # Create datasets
    train_dataset = SPairDataset(
        pair_ann_path, layout_path, image_path,
        dataset_size, pck_alpha, datatype='trn'
    )
    val_dataset = SPairDataset(
        pair_ann_path, layout_path, image_path,
        dataset_size, pck_alpha, datatype='val'
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        collate_fn=custom_collate_fn
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=custom_collate_fn
    )

    print(f"Train samples: {len(train_dataset)}")
    print(f"Val samples: {len(val_dataset)}\n")

    # Optimizer
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate,
        weight_decay=1e-4
    )

    # Training loop
    best_pck = 0.0
    history = {
        'train_loss': [],
        'val_pck_005': [],
        'val_pck_010': [],
        'val_pck_020': []
    }

    for epoch in range(num_epochs):
        print(f"\n{'='*70}")
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print(f"{'='*70}")

        # Train with batch limit
        train_loss = train_epoch(
            model, train_loader, optimizer, device, model_type, patch_size,
            debug_batches=debug_batches
        )

        # Validate with batch limit (half of train batches)
        val_limit = debug_batches // 2 if debug_batches else None
        val_pck_dict = validate_epoch(
            model, val_loader, device, model_type, patch_size,
            thresholds=[0.05, 0.10, 0.20],
            debug_batches=val_limit
        )

        # Record history
        history['train_loss'].append(train_loss)
        history['val_pck_005'].append(val_pck_dict[0.05])
        history['val_pck_010'].append(val_pck_dict[0.10])
        history['val_pck_020'].append(val_pck_dict[0.20])

        # Print results
        print(f"\n{'─'*70}")
        print(f"Epoch {epoch + 1} Results:")
        print(f"  Train Loss:   {train_loss:.4f}")
        print(f"  Val PCK@0.05: {val_pck_dict[0.05]:.2f}%")
        print(f"  Val PCK@0.10: {val_pck_dict[0.10]:.2f}%")
        print(f"  Val PCK@0.20: {val_pck_dict[0.20]:.2f}%")

        # Save best model
        if val_pck_dict[0.10] > best_pck:
            best_pck = val_pck_dict[0.10]
            save_path = f'best_{model_type}_L{num_layers_to_unfreeze}_head{int(head)}_ep{epoch+1}.pth'

            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_pck': val_pck_dict,
                'train_loss': train_loss,
                'history': history,
                'config': {
                    'model_type': model_type,
                    'num_layers': num_layers_to_unfreeze,
                    'head': head,
                    'learning_rate': learning_rate,
                    'batch_size': batch_size
                }
            }, save_path)
            print(f"  ✓ New best model saved! ({save_path})")

        print(f"{'─'*70}")

    print(f"\n{'='*70}")
    print(f"Training Complete!")
    print(f"Best Val PCK@0.10: {best_pck:.2f}%")
    print(f"{'='*70}\n")

    return history

In [ ]:
def plot_training_history(history):
    """
    Plot training loss and validation PCK
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss
    ax1.plot(history['train_loss'], marker='o')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Training Loss')
    ax1.set_title('Training Loss over Epochs')
    ax1.grid(True)

    # PCK
    ax2.plot(history['val_pck_005'], marker='o', label='PCK@0.05')
    ax2.plot(history['val_pck_010'], marker='s', label='PCK@0.10')
    ax2.plot(history['val_pck_020'], marker='^', label='PCK@0.20')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('PCK Score')
    ax2.set_title('Validation PCK over Epochs')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150)
    plt.show()

In [ ]:
# ========== CONFIGURATION for training ==========
# Choose model: 'dinov2', 'dinov3', or 'sam'
MODEL_NAME = 'dinov2'

# Training hyperparameters
NUM_EPOCHS = 5
BATCH_SIZE = 8
LEARNING_RATE = 5e-5
NUM_LAYERS_TO_UNFREEZE = 3
HEAD = False  # True = also unfreeze norm/neck layers

# Debug mode (set to None for full training)
DEBUG_BATCHES = 5  # Use None for full training, or a number like 5 for quick test

# ========== SETUP MODEL ==========
device = 'cuda' if torch.cuda.is_available() else 'cpu'

if 'dinov2' in MODEL_NAME:
    model = dinov2_vitb14
    patch_size = 14
elif 'dinov3' in MODEL_NAME:
    model = dinov3_vitb16
    patch_size = 16
elif 'sam' in MODEL_NAME:
    model = sam
    patch_size = 16
else:
    raise ValueError(f"Unknown model: {MODEL_NAME}. Use 'dinov2', 'dinov3', or 'sam'.")

print(f"Model: {MODEL_NAME} | Device: {device} | Patch size: {patch_size}")

In [ ]:
# ========== RUN TRAINING ==========
history = finetune(
    model=model,
    num_layers_to_unfreeze=NUM_LAYERS_TO_UNFREEZE,
    pair_ann_path=pair_ann_path,
    layout_path=layout_path,
    image_path=image_path,
    model_type=MODEL_NAME,
    patch_size=patch_size,
    head=HEAD,
    dataset_size='large',
    pck_alpha=0.1,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    device=device,
    debug_batches=DEBUG_BATCHES
)

# Plot results
plot_training_history(history)